# LTX-Video 0.9.8 distilled (2B) — пробный инференс

Модель из `docs/design.md`: `ltxv-2b-0.9.8-distilled`, самая быстрая в семействе.

Отдельного diffusers-репозитория у неё нет — 0.9.8 лежит в `Lightricks/LTX-Video`
одним файлом `.safetensors` (6,3 ГБ). Поэтому трансформер и VAE грузятся через
`from_single_file`, а текстовый энкодер (T5-XXL), токенизатор и планировщик
берутся из diffusers-репозиториев.

Пайплайн — `LTXConditionPipeline`: принимает условие и картинкой (`image=`),
и куском видео (`video=` + `frame_index`), что и нужно для склейки кусков по хвосту.

**Ядро:** system python (`/usr/bin/python3`, torch 2.14+cu130) — пакеты стоят в нём.
В `/venv/main` другой torch и нет diffusers, ядро туда не переключать.
Веса кэшируются в `$HF_HOME` = `/workspace/.hf_home`.

In [ ]:
!pip install -q -U diffusers transformers accelerate
# без этих трёх T5-токенизатор не читается в transformers 5.x, а mp4 не записывается
!pip install -q tiktoken sentencepiece protobuf imageio imageio-ffmpeg

In [ ]:
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

## Загрузка пайплайна

Первый запуск качает 6,3 ГБ (сам чекпойнт) плюс T5-XXL, дальше грузится из кэша
примерно за 110 с.

In [ ]:
import time, torch
from diffusers import (LTXConditionPipeline, LTXVideoTransformer3DModel,
                       AutoencoderKLLTXVideo, FlowMatchEulerDiscreteScheduler)
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from transformers import T5EncoderModel, T5Tokenizer
from diffusers.utils import load_image, export_to_video

CKPT = "https://huggingface.co/Lightricks/LTX-Video/blob/main/ltxv-2b-0.9.8-distilled.safetensors"
DT = torch.bfloat16

t0 = time.time()
transformer  = LTXVideoTransformer3DModel.from_single_file(CKPT, dtype=DT)
vae          = AutoencoderKLLTXVideo.from_single_file(CKPT, dtype=DT)
text_encoder = T5EncoderModel.from_pretrained("Lightricks/LTX-Video", subfolder="text_encoder", dtype=DT)
tokenizer    = T5Tokenizer.from_pretrained("Lightricks/LTX-Video", subfolder="tokenizer")
scheduler    = FlowMatchEulerDiscreteScheduler.from_pretrained(
    "Lightricks/LTX-Video-0.9.8-13B-distilled", subfolder="scheduler")

pipe = LTXConditionPipeline(
    vae=vae, text_encoder=text_encoder, tokenizer=tokenizer,
    transformer=transformer, scheduler=scheduler,
).to("cuda")
print(f"загрузка: {time.time() - t0:.1f}s")

## Генерация

Distilled-модель рассчитана на малое число шагов и работает **без CFG**
(`guidance_scale=1.0`) — с большими значениями качество только портится.

Поэтому `negative_prompt` здесь не задаётся: в `LTXConditionPipeline` флаг
`do_classifier_free_guidance` — это буквально `guidance_scale > 1.0`, и при 1.0
негативный промпт даже не кодируется, а молча игнорируется. Чтобы он заработал,
нужен `guidance_scale > 1`, но это удваивает стоимость шага (латент идёт через
модель дважды) и ломает distilled-режим — то есть прямо противоречит цели G < L.

Ограничения на размеры: ширина и высота кратны 32, число кадров вида `8k + 1`.

In [ ]:
image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/guitar-man.png"
)
cond = LTXVideoCondition(image=image, frame_index=0)

FPS = 24
frames = pipe(
    conditions=[cond],
    prompt="A man with short gray hair plays a red electric guitar.",
    width=704, height=480, num_frames=97,
    num_inference_steps=8, guidance_scale=1.0,
    generator=torch.Generator("cuda").manual_seed(0),
).frames[0]

export_to_video(frames, "output.mp4", fps=FPS)
print("кадров:", len(frames))

In [ ]:
from IPython.display import Video
Video("output.mp4", embed=True, width=704)

## Замер G/L

Главный критерий из дизайн-дока: кусок длиной **L** секунд должен генерироваться
за **G < L**, цель — G/L ≈ 0,5–0,7.

Первый прогон медленнее из-за прогрева, поэтому меряем со второго.

In [ ]:
def bench(width, height, num_frames, steps=8, runs=3):
    torch.cuda.reset_peak_memory_stats()
    times = []
    for i in range(runs):
        t0 = time.time()
        pipe(
            conditions=[cond],
            prompt="A man with short gray hair plays a red electric guitar.",
            width=width, height=height, num_frames=num_frames,
            num_inference_steps=steps, guidance_scale=1.0,
            generator=torch.Generator("cuda").manual_seed(0),
        )
        times.append(time.time() - t0)
    G = min(times[1:]) if runs > 1 else times[0]      # без прогревочного прогона
    L = num_frames / FPS
    print(f"{width}x{height}, {num_frames} кадров ({L:.1f}s видео), {steps} шагов: "
          f"G={G:.2f}s  G/L={G / L:.2f}  VRAM={torch.cuda.max_memory_allocated() / 1e9:.1f} ГБ")
    return G, L

bench(704, 480, 97)

## Циклическая генерация

Шаг 0 из `docs/design.md`: цепочка кусков, где каждый продолжает предыдущий.

Как устроено продолжение. `LTXVideoCondition(video=tail, frame_index=0)` кладёт
латенты хвоста в **начало** нового куска, то есть кусок начинается ровно теми
кадрами, на которые его обусловили. Поэтому перекрытие надо отрезать из вывода:
полезного прироста за кусок получается `chunk_frames - overlap` кадров.

Два ограничения пайплайна, оба вида `8k + 1` (временная компрессия VAE = 8):
длина куска и длина хвоста. `overlap=9` даёт 2 латентных кадра условия —
это минимум, при котором модель видит направление движения, а не одну статичную картинку.

Считаем G относительно **прироста**, а не всей длины куска: в эфир уходит только новое.

In [ ]:
import time
import numpy as np
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition

def generate_chain(start_image, prompt_for, n_chunks,
                   chunk_frames=97, overlap=9, width=704, height=480,
                   steps=8, seed=0, fps=FPS, verbose=True):
    """Цепочка кусков, каждый продолжает хвост предыдущего.

    prompt_for: f(i) -> str, промпт для куска i (сюда позже придёт чат).
    Возвращает (кадры, список словарей со статистикой по кускам).
    """
    for name, v in (("chunk_frames", chunk_frames), ("overlap", overlap)):
        if (v - 1) % 8 != 0:
            raise ValueError(f"{name} должен быть вида 8k+1, получено {v}")
    if overlap >= chunk_frames:
        raise ValueError("overlap должен быть меньше chunk_frames")

    frames, stats, tail = [], [], None
    for i in range(n_chunks):
        cond = (LTXVideoCondition(image=start_image, frame_index=0) if tail is None
                else LTXVideoCondition(video=tail, frame_index=0))
        t0 = time.time()
        out = pipe(
            conditions=[cond],
            prompt=prompt_for(i),
            width=width, height=height, num_frames=chunk_frames,
            num_inference_steps=steps, guidance_scale=1.0,
            generator=torch.Generator("cuda").manual_seed(seed + i),
        ).frames[0]
        G = time.time() - t0

        new = out if tail is None else out[overlap:]   # перекрытие уже показано
        frames.extend(new)
        tail = out[-overlap:]

        L = len(new) / fps
        mean_rgb = np.asarray(new[-1], dtype=np.float32).mean(axis=(0, 1))
        stats.append({"i": i, "G": G, "L": L, "ratio": G / L, "mean_rgb": mean_rgb})
        if verbose:
            print(f"кусок {i:3d}  G={G:5.2f}s  L={L:4.2f}s  G/L={G / L:4.2f}  "
                  f"RGB={mean_rgb.round(1)}  всего {len(frames) / fps:6.1f}s", flush=True)
    return frames, stats

### Прогон

Для начала 8 кусков на одном промпте — этого хватает, чтобы увидеть стыки.
Полный замер из дизайн-дока — 20–30 кусков, чтобы поймать деградацию.

In [ ]:
PROMPT = ("A man with short gray hair plays a red electric guitar on a small stage, "
          "warm stage lights, smooth camera movement")

torch.cuda.reset_peak_memory_stats()
frames, stats = generate_chain(
    start_image=image,
    prompt_for=lambda i: PROMPT,
    n_chunks=8,
)
export_to_video(frames, "chain.mp4", fps=FPS)

ratios = [s["ratio"] for s in stats[1:]]           # без первого, он прогревочный
print(f"\nG/L: медиана {np.median(ratios):.2f}, худший {max(ratios):.2f}")
print(f"длительность: {len(frames) / FPS:.1f}s, пик VRAM {torch.cuda.max_memory_allocated() / 1e9:.1f} ГБ")

drift = np.linalg.norm(stats[-1]["mean_rgb"] - stats[0]["mean_rgb"])
print(f"снос средней яркости от первого куска к последнему: {drift:.1f} (из 255)")

In [ ]:
from IPython.display import Video
Video("chain.mp4", embed=True, width=704)

### Смена промпта по ходу

Имитация чата: на середине цепочки приходит запрос. Видно и как быстро модель
подхватывает новый промпт, и насколько резким выходит переход — по дизайн-доку
это повод для намеренной смены сцены, а не склейки по хвосту.

In [ ]:
frames2, stats2 = generate_chain(
    start_image=image,
    prompt_for=lambda i: PROMPT if i < 4 else (
        "The same man on stage, the lights turn deep blue, fog rolls across the floor"),
    n_chunks=8,
    seed=100,
)
export_to_video(frames2, "chain_prompt_switch.mp4", fps=FPS)
Video("chain_prompt_switch.mp4", embed=True, width=704)

### Что смотреть в результате

* **G/L** — держится ли запас. Считается от прироста `chunk_frames - overlap`, а не
  от всей длины куска: в эфир уходит только новое.
* **Стыки** — рывки на границах кусков, каждые `chunk_frames - overlap` кадров.
* **Деградация** — уход цвета и «плывущая» картинка к концу.

Замерено на цепочке из 30 кусков (110 с видео): G/L медиана **0,50**, стыки
не выделяются на фоне обычного движения (межкадровая разница на границах даже
меньше фоновой), расплывания картинки нет. Реальная проблема другая — сцена
монотонно уползает, камера наезжает до крупного плана. Лечится принудительной
сменой сцены раз в 10–20 кусков.

## Разрешение и две бесплатные оптимизации

**Выгрузка T5 после кодирования промпта** срезает пик с 19,3 до 9,7 ГБ. Схема:
поднять T5 на CPU, закодировать промпты, удалить, а пайплайн собрать с
`text_encoder=None` — `encode_prompt` обращается к энкодеру только когда
`prompt_embeds is None`. Для стрима это и так правильно: промпт меняется редко,
эмбеддинги готовятся заранее. Подробнее в `test_LTXV_13B.ipynb`.

**`torch.compile` на 2B не даёт ничего** — проверено, разница в пределах шума:

| разрешение | G | G/L в цепочке | пик VRAM | с compile |
|---|---|---|---|---|
| 704×480 (здесь) | 1,77 с | **0,48** | 9,7 ГБ | 1,92 с — хуже |
| **768×512** | 2,22 с | **0,61** | 10,2 ГБ | 2,28 с — хуже |
| 960×544 | 3,21 с | **0,88** | 11,5 ГБ | 3,16 с |
| 1152×640 | 5,29 с | 1,44 | 13,6 ГБ | 5,08 с |

(На 13B `compile` давал −25%, но там он компенсировал накладные расходы fp8-квантования
torchao, а не ускорял модель как таковую.)

Значит, запас тратим на разрешение. **Рабочая точка — 768×512**: G/L 0,61, памяти
10 ГБ из 32. Прирост детализации виден глазами — на 704×480 колки грифа смазаны,
на 960×544 читается логотип.

Чтобы перевести цепочку на 768×512, достаточно передать размеры в `generate_chain`:

In [ ]:
# frames, stats = generate_chain(
#     start_image=image,
#     prompt_for=lambda i: PROMPT,
#     n_chunks=8,
#     width=768, height=512,
# )